# ⭐ Day 62: Feature Engineering Mastery

## Creating Better Features for Stronger Models

**Python & AI Learning Path — Day 62 of 369**


## 📊 Introduction

Welcome to Day 62 of our 369-day Python & AI Learning Path! Today we dive deep into **feature engineering** — the art and science of transforming raw data into features that better represent the underlying problem to machine learning algorithms.

Feature engineering is often cited as the single most important factor in building high-performing machine learning models. Kaggle competition winners consistently report that well-engineered features contribute more to their success than algorithm selection or hyperparameter tuning. A model with mediocre algorithms but excellent features will almost always outperform a model with state-of-the-art algorithms but poor features.

Why does feature engineering matter so much? Machine learning algorithms learn patterns from data, but they can only discover patterns that are present in the features you provide. Clever feature engineering creates representations that make these patterns more explicit, easier to learn, and more robust to noise. It bridges the gap between raw data and the insights your model needs to make accurate predictions.

Today, we'll work with two classic datasets — the **Titanic** dataset (classification) and the **House Prices** dataset (regression) — to demonstrate practical, powerful feature engineering techniques. We'll show you how to create features that dramatically improve model performance, with before-and-after comparisons that prove the value of each technique.

By the end of this notebook, you'll have a comprehensive toolkit of feature engineering strategies and the mindset to apply them creatively to any dataset you encounter. Let's build better features! 🚀


## 📑 Table of Contents

1. [Why Feature Engineering Matters More Than Algorithms](#1-why-feature-engineering-matters-more-than-algorithms)
2. [Feature Engineering Mindset and Best Practices](#2-feature-engineering-mindset-and-best-practices)
3. [Numerical Feature Engineering](#3-numerical-feature-engineering)
   - 3.1 Binning / Discretization
   - 3.2 Scaling & Normalization
   - 3.3 Log & Power Transformations
   - 3.4 Creating Interaction Features
   - 3.5 Polynomial Features
4. [Categorical Feature Engineering](#4-categorical-feature-engineering)
   - 4.1 One-Hot vs Target/Mean Encoding
   - 4.2 Frequency Encoding
   - 4.3 Grouping Rare Categories
5. [Date & Time Feature Engineering](#5-date--time-feature-engineering)
6. [Text Feature Engineering](#6-text-feature-engineering)
7. [Domain-Specific Feature Creation](#7-domain-specific-feature-creation)
8. [Feature Selection Techniques](#8-feature-selection-techniques)
9. [End-to-End Feature Engineering Pipeline Example](#9-end-to-end-feature-engineering-pipeline-example)
10. [Hands-On Exercises](#-hands-on-exercises)
11. [Solutions & Best Practices](#-solutions--best-practices-review-after-attempting)


## Setup & Imports

Let's start by importing the libraries we'll need and loading our datasets.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report
from sklearn.feature_selection import SelectKBest, f_classif, f_regression, RFE, SelectFromModel
import warnings
warnings.filterwarnings('ignore')

# Set style for beautiful plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


In [2]:
# Load Titanic dataset
titanic_url = "https://raw.githubusercontent.com/datasciencedoc/data/master/titanic_train.csv"
titanic = pd.read_csv(titanic_url)

# Load House Prices dataset (using a sample for demonstration)
house_url = "https://raw.githubusercontent.com/datasciencedoc/data/master/house_prices_train.csv"
house = pd.read_csv(house_url)

print(f"📊 Titanic Dataset: {titanic.shape[0]} rows, {titanic.shape[1]} columns")
print(f"🏠 House Prices Dataset: {house.shape[0]} rows, {house.shape[1]} columns")
print("\n🔍 Titanic columns:", titanic.columns.tolist())


HTTPError: HTTP Error 404: Not Found

## 1. Why Feature Engineering Matters More Than Algorithms

Before we dive into techniques, let's establish a baseline to prove that feature engineering beats algorithm tuning.


In [ ]:
# Baseline comparison: Raw features vs Engineered features

# Prepare raw Titanic data (minimal preprocessing)
titanic_raw = titanic.copy()
titanic_raw = titanic_raw[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']].copy()
titanic_raw['Sex'] = titanic_raw['Sex'].map({'male': 0, 'female': 1})
titanic_raw['Age'] = titanic_raw['Age'].fillna(titanic_raw['Age'].median())

X_raw = titanic_raw.drop('Survived', axis=1)
y = titanic_raw['Survived']

# Baseline models on raw features
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

print("📊 BASELINE PERFORMANCE (Raw Features Only):\n")
baseline_scores = {}
for name, model in models.items():
    scores = cross_val_score(model, X_raw, y, cv=5, scoring='accuracy')
    baseline_scores[name] = scores.mean()
    print(f"  {name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

print("\n💡 Notice: These are decent scores, but we can do MUCH better with feature engineering!")


## 2. Feature Engineering Mindset and Best Practices

Before writing code, let's internalize the key principles:

1. **Understand the domain** — Business knowledge creates the best features
2. **Start simple, then iterate** — Don't over-engineer from day one
3. **Visualize before and after** — Always measure the impact
4. **Beware of data leakage** — Never use future information or target variable inappropriately
5. **Document your reasoning** — Every feature should have a "why"
6. **Test rigorously** — Use cross-validation to validate improvements


## 3. Numerical Feature Engineering

### 3.1 Binning / Discretization

💡 **Why This Feature Helps:** Continuous variables can hide important thresholds. Age groups (child, adult, senior) may have very different survival rates on the Titanic. Binning captures these non-linear relationships.


In [ ]:
# Binning Age on Titanic dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before: Raw Age distribution
sns.histplot(data=titanic, x='Age', hue='Survived', bins=30, kde=True, ax=axes[0])
axes[0].set_title('Before: Raw Age Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')

# Create age bins
titanic['AgeBin'] = pd.cut(titanic['Age'], bins=[0, 12, 18, 35, 60, 100], 
                            labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])

# After: Binned Age survival rates
age_survival = titanic.groupby('AgeBin')['Survived'].mean().reset_index()
sns.barplot(data=age_survival, x='AgeBin', y='Survived', ax=axes[1])
axes[1].set_title('After: Survival Rate by Age Group', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("✅ Binning reveals clear patterns: Children had higher survival rates!")
print(age_survival)


### 3.2 Scaling & Normalization

💡 **Why This Feature Helps:** Algorithms like logistic regression, SVM, and neural networks are sensitive to feature scales. Scaling ensures all features contribute equally and improves convergence.


In [ ]:
# Scaling demonstration with House Prices dataset
numeric_features = ['GrLivArea', 'TotalBsmtSF', '1stFlrSF', 'GarageArea', 'SalePrice']
house_numeric = house[numeric_features].dropna()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Before scaling
axes[0, 0].scatter(house_numeric['GrLivArea'], house_numeric['TotalBsmtSF'], alpha=0.5)
axes[0, 0].set_title('Before Scaling: Raw Values', fontweight='bold')
axes[0, 0].set_xlabel('GrLivArea')
axes[0, 0].set_ylabel('TotalBsmtSF')

# Standard Scaling (Z-score)
scaler = StandardScaler()
scaled = scaler.fit_transform(house_numeric[['GrLivArea', 'TotalBsmtSF']])
axes[0, 1].scatter(scaled[:, 0], scaled[:, 1], alpha=0.5, color='green')
axes[0, 1].set_title('After StandardScaler (Z-score)', fontweight='bold')
axes[0, 1].set_xlabel('GrLivArea (scaled)')
axes[0, 1].set_ylabel('TotalBsmtSF (scaled)')

# Min-Max Scaling
minmax = MinMaxScaler()
minmax_scaled = minmax.fit_transform(house_numeric[['GrLivArea', 'TotalBsmtSF']])
axes[1, 0].scatter(minmax_scaled[:, 0], minmax_scaled[:, 1], alpha=0.5, color='orange')
axes[1, 0].set_title('After MinMaxScaler [0,1]', fontweight='bold')
axes[1, 0].set_xlabel('GrLivArea (minmax)')
axes[1, 0].set_ylabel('TotalBsmtSF (minmax)')

# Distribution comparison
axes[1, 1].hist(house_numeric['GrLivArea'], bins=50, alpha=0.5, label='Original', density=True)
axes[1, 1].hist(scaled[:, 0], bins=50, alpha=0.5, label='StandardScaled', density=True)
axes[1, 1].set_title('Distribution: Original vs Scaled', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

print("✅ Scaling preserves relationships but puts features on comparable scales!")


### 3.3 Log & Power Transformations

💡 **Why This Feature Helps:** Many real-world features (income, house prices, population) follow power-law or right-skewed distributions. Log transforms compress the long tail, making distributions more normal and improving linear model performance.


In [ ]:
# Log transformation on House Prices
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original SalePrice (right-skewed)
axes[0, 0].hist(house['SalePrice'].dropna(), bins=50, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Original SalePrice (Right-Skewed)', fontweight='bold')
axes[0, 0].set_xlabel('SalePrice')

# Log-transformed SalePrice
house['LogSalePrice'] = np.log1p(house['SalePrice'])
axes[0, 1].hist(house['LogSalePrice'].dropna(), bins=50, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Log-Transformed SalePrice (More Normal)', fontweight='bold')
axes[0, 1].set_xlabel('log(SalePrice + 1)')

# Original GrLivArea
axes[1, 0].hist(house['GrLivArea'].dropna(), bins=50, color='salmon', edgecolor='black')
axes[1, 0].set_title('Original GrLivArea', fontweight='bold')

# Log-transformed GrLivArea
house['LogGrLivArea'] = np.log1p(house['GrLivArea'])
axes[1, 1].hist(house['LogGrLivArea'].dropna(), bins=50, color='gold', edgecolor='black')
axes[1, 1].set_title('Log-Transformed GrLivArea', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Log transformation reduces skewness and makes data more Gaussian-like!")
print(f"   Original SalePrice skew: {house['SalePrice'].skew():.2f}")
print(f"   Log SalePrice skew: {house['LogSalePrice'].skew():.2f}")


### 3.4 Creating Interaction Features

💡 **Why This Feature Helps:** Sometimes the combined effect of two features is more predictive than either alone. For example, `FamilySize` (SibSp + Parch + 1) on the Titanic captures the idea that traveling alone vs. with family affected survival differently.


In [ ]:
# Interaction features on Titanic
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['IsAlone'] = (titanic['FamilySize'] == 1).astype(int)

# Fare per person
titanic['FarePerPerson'] = titanic['Fare'] / titanic['FamilySize']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Family Size vs Survival
fam_survival = titanic.groupby('FamilySize')['Survived'].mean().reset_index()
sns.barplot(data=fam_survival, x='FamilySize', y='Survived', ax=axes[0])
axes[0].set_title('Survival Rate by Family Size', fontweight='bold')
axes[0].set_ylim(0, 1)

# Alone vs Not Alone
alone_survival = titanic.groupby('IsAlone')['Survived'].mean().reset_index()
alone_survival['IsAlone'] = alone_survival['IsAlone'].map({0: 'With Family', 1: 'Alone'})
sns.barplot(data=alone_survival, x='IsAlone', y='Survived', ax=axes[1])
axes[1].set_title('Alone vs With Family', fontweight='bold')
axes[1].set_ylim(0, 1)

# Fare per person distribution
sns.boxplot(data=titanic, x='Survived', y='FarePerPerson', ax=axes[2])
axes[2].set_title('Fare Per Person by Survival', fontweight='bold')
axes[2].set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("✅ Interaction features reveal powerful patterns!")
print(f"   Alone survival rate: {titanic[titanic['IsAlone']==1]['Survived'].mean():.3f}")
print(f"   With family survival rate: {titanic[titanic['IsAlone']==0]['Survived'].mean():.3f}")


### 3.5 Polynomial Features

💡 **Why This Feature Helps:** Linear models can only learn linear relationships. Polynomial features allow linear models to capture curves and interactions without changing the algorithm.


In [ ]:
# Polynomial features demonstration
from sklearn.preprocessing import PolynomialFeatures

# Use a subset of House Prices features
poly_features = ['GrLivArea', 'TotalBsmtSF']
house_poly = house[poly_features + ['SalePrice']].dropna()

X_poly = house_poly[poly_features]
y_poly = np.log1p(house_poly['SalePrice'])

# Create polynomial features (degree 2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_transformed = poly.fit_transform(X_poly)
feature_names = poly.get_feature_names_out(poly_features)

print(f"📊 Original features: {X_poly.shape[1]}")
print(f"📊 Polynomial features: {X_poly_transformed.shape[1]}")
print(f"📋 New feature names: {feature_names.tolist()}")

# Compare Linear Regression performance
X_train, X_test, y_train, y_test = train_test_split(X_poly, y_poly, test_size=0.2, random_state=42)
X_train_p, X_test_p, _, _ = train_test_split(X_poly_transformed, y_poly, test_size=0.2, random_state=42)

# Baseline (linear)
lr = LinearRegression()
lr.fit(X_train, y_train)
baseline_rmse = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))

# With polynomial features
lr_poly = Ridge(alpha=1.0)
lr_poly.fit(X_train_p, y_train)
poly_rmse = np.sqrt(mean_squared_error(y_test, lr_poly.predict(X_test_p)))

print(f"\n📈 Performance Comparison:")
print(f"   Linear features RMSE: {baseline_rmse:.4f}")
print(f"   Polynomial features RMSE: {poly_rmse:.4f}")
print(f"   Improvement: {((baseline_rmse - poly_rmse) / baseline_rmse * 100):.1f}%")


## 4. Categorical Feature Engineering

### 4.1 One-Hot vs Target/Mean Encoding

💡 **Why This Feature Helps:** Machine learning models need numbers. How we encode categories dramatically affects what patterns the model can learn. One-hot is safe but expands dimensionality; target encoding is compact but risks overfitting.


In [ ]:
# Encoding comparison on Titanic Embarked feature
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original categorical
embarked_survival = titanic.groupby('Embarked')['Survived'].mean().reset_index()
sns.barplot(data=embarked_survival, x='Embarked', y='Survived', ax=axes[0])
axes[0].set_title('Survival Rate by Embarked', fontweight='bold')
axes[0].set_ylim(0, 1)

# One-Hot Encoding
embarked_dummies = pd.get_dummies(titanic['Embarked'], prefix='Embarked')
print("📊 One-Hot Encoded columns:")
print(embarked_dummies.head())

# Target Encoding (mean survival rate per category)
target_means = titanic.groupby('Embarked')['Survived'].mean()
titanic['Embarked_TargetEnc'] = titanic['Embarked'].map(target_means)

axes[1].bar(target_means.index, target_means.values, color=['coral', 'skyblue', 'lightgreen'])
axes[1].set_title('Target Encoding Values', fontweight='bold')
axes[1].set_ylabel('Mean Survival Rate')

# Frequency Encoding
freq_map = titanic['Embarked'].value_counts().to_dict()
titanic['Embarked_FreqEnc'] = titanic['Embarked'].map(freq_map)

axes[2].bar(freq_map.keys(), freq_map.values(), color=['coral', 'skyblue', 'lightgreen'])
axes[2].set_title('Frequency Encoding Values', fontweight='bold')
axes[2].set_ylabel('Frequency Count')

plt.tight_layout()
plt.show()

print("\n✅ One-Hot: Safe, no information leakage, but increases dimensions")
print("✅ Target Encoding: Compact, captures relationship with target, but risk of overfitting")
print("✅ Frequency Encoding: Simple, captures category popularity")


### 4.2 Frequency Encoding

💡 **Why This Feature Helps:** Categories that appear frequently vs. rarely may have different behaviors. Frequency encoding captures this information in a single numeric column.


In [ ]:
# Frequency encoding on House Prices - Neighborhood
neighborhood_freq = house['Neighborhood'].value_counts()
house['Neighborhood_Freq'] = house['Neighborhood'].map(neighborhood_freq)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency distribution
axes[0].bar(range(len(neighborhood_freq)), neighborhood_freq.values, color='teal')
axes[0].set_title('Neighborhood Frequency Distribution', fontweight='bold')
axes[0].set_xlabel('Neighborhood (sorted by frequency)')
axes[0].set_ylabel('Count')

# Frequency vs SalePrice
freq_price = house.groupby('Neighborhood_Freq')['SalePrice'].mean().reset_index()
axes[1].scatter(freq_price['Neighborhood_Freq'], freq_price['SalePrice'], alpha=0.7, color='purple')
axes[1].set_title('Neighborhood Frequency vs Avg SalePrice', fontweight='bold')
axes[1].set_xlabel('Neighborhood Frequency')
axes[1].set_ylabel('Average SalePrice')

plt.tight_layout()
plt.show()

print("✅ Frequency encoding can reveal whether common vs rare neighborhoods differ in price!")


### 4.3 Grouping Rare Categories

💡 **Why This Feature Helps:** Rare categories can cause overfitting and create sparse one-hot encoded columns. Grouping them into an "Other" category reduces noise and improves generalization.


In [ ]:
# Group rare neighborhoods in House Prices
threshold = 50  # Minimum count to keep as separate category
neighborhood_counts = house['Neighborhood'].value_counts()
rare_neighborhoods = neighborhood_counts[neighborhood_counts < threshold].index

house['Neighborhood_Grouped'] = house['Neighborhood'].replace(rare_neighborhoods, 'Other')

print(f"📊 Original neighborhoods: {house['Neighborhood'].nunique()}")
print(f"📊 After grouping rare: {house['Neighborhood_Grouped'].nunique()}")
print(f"📊 Categories grouped into 'Other': {len(rare_neighborhoods)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before grouping
house['Neighborhood'].value_counts().plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Before: All Neighborhoods', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# After grouping
house['Neighborhood_Grouped'].value_counts().plot(kind='bar', ax=axes[1], color='lightgreen')
axes[1].set_title('After: Rare Categories Grouped', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Grouping rare categories reduces dimensionality and prevents overfitting!")


## 5. Date & Time Feature Engineering

💡 **Why This Feature Helps:** Dates contain rich information (seasonality, trends, cyclical patterns) that raw timestamps hide. Extracting year, month, day, and cyclical features captures these patterns.


In [ ]:
# Create a sample date feature for demonstration
np.random.seed(42)
dates = pd.date_range(start='2006-01-01', end='2010-12-31', freq='D')
sample_dates = np.random.choice(dates, size=len(house))
house['SaleDate'] = sample_dates

# Extract date components
house['SaleYear'] = house['SaleDate'].dt.year
house['SaleMonth'] = house['SaleDate'].dt.month
house['SaleDay'] = house['SaleDate'].dt.day
house['SaleQuarter'] = house['SaleDate'].dt.quarter
house['SaleDayOfWeek'] = house['SaleDate'].dt.dayofweek
house['SaleIsWeekend'] = (house['SaleDayOfWeek'] >= 5).astype(int)

# Cyclical encoding for month (preserves circular nature of months)
house['SaleMonth_Sin'] = np.sin(2 * np.pi * house['SaleMonth'] / 12)
house['SaleMonth_Cos'] = np.cos(2 * np.pi * house['SaleMonth'] / 12)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Monthly pattern
monthly_sales = house.groupby('SaleMonth')['SalePrice'].mean()
axes[0, 0].plot(monthly_sales.index, monthly_sales.values, marker='o', color='navy')
axes[0, 0].set_title('Average SalePrice by Month', fontweight='bold')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Average SalePrice')

# Yearly trend
yearly_sales = house.groupby('SaleYear')['SalePrice'].mean()
axes[0, 1].plot(yearly_sales.index, yearly_sales.values, marker='s', color='darkgreen')
axes[0, 1].set_title('Average SalePrice by Year', fontweight='bold')
axes[0, 1].set_xlabel('Year')

# Weekend vs Weekday
weekend_sales = house.groupby('SaleIsWeekend')['SalePrice'].mean()
weekend_sales.index = ['Weekday', 'Weekend']
axes[1, 0].bar(weekend_sales.index, weekend_sales.values, color=['coral', 'skyblue'])
axes[1, 0].set_title('Weekday vs Weekend Sales', fontweight='bold')

# Cyclical encoding visualization
axes[1, 1].scatter(house['SaleMonth_Sin'], house['SaleMonth_Cos'], alpha=0.5, c=house['SaleMonth'], cmap='hsv')
axes[1, 1].set_title('Cyclical Month Encoding (Circle)', fontweight='bold')
axes[1, 1].set_xlabel('sin(month)')
axes[1, 1].set_ylabel('cos(month)')
axes[1, 1].set_aspect('equal')

plt.tight_layout()
plt.show()

print("✅ Date features capture seasonality, trends, and cyclical patterns!")
print("✅ Cyclical encoding ensures December (12) and January (1) are close in feature space!")


## 6. Text Feature Engineering

💡 **Why This Feature Helps:** Text fields like names, descriptions, and addresses contain valuable structured information. Simple extraction techniques can capture this without complex NLP.


In [ ]:
# Text feature extraction from Titanic Names
titanic['Name_Length'] = titanic['Name'].str.len()

# Extract Title from Name
titanic['Title'] = titanic['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Group rare titles
title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Major': 'Rare',
    'Mlle': 'Miss',
    'Countess': 'Rare',
    'Ms': 'Miss',
    'Lady': 'Rare',
    'Jonkheer': 'Rare',
    'Don': 'Rare',
    'Dona': 'Rare',
    'Mme': 'Mrs',
    'Capt': 'Rare',
    'Sir': 'Rare'
}
titanic['Title_Grouped'] = titanic['Title'].map(title_mapping)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Name length vs survival
name_len_survival = titanic.groupby(pd.cut(titanic['Name_Length'], bins=5))['Survived'].mean()
name_len_survival.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Survival Rate by Name Length', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Title distribution
title_counts = titanic['Title_Grouped'].value_counts()
axes[1].pie(title_counts.values, labels=title_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Title Distribution', fontweight='bold')

# Title vs survival
title_survival = titanic.groupby('Title_Grouped')['Survived'].mean().sort_values(ascending=False)
axes[2].bar(title_survival.index, title_survival.values, color='mediumseagreen')
axes[2].set_title('Survival Rate by Title', fontweight='bold')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Text features extracted: Name Length, Title")
print("✅ Titles reveal social status — a powerful predictor of survival!")
print(title_survival)


## 7. Domain-Specific Feature Creation

💡 **Why This Feature Helps:** Domain knowledge creates the most impactful features. Understanding the problem context (e.g., Titanic's "women and children first" policy) leads to features algorithms would never discover alone.


In [ ]:
# Domain-specific features for Titanic

# 1. Women and Children First policy
titanic['IsWomanOrChild'] = ((titanic['Sex'] == 'female') | (titanic['Age'] < 16)).astype(int)

# 2. Deck from Cabin (first letter)
titanic['Deck'] = titanic['Cabin'].str[0]
titanic['Deck'] = titanic['Deck'].fillna('Unknown')
titanic['HasCabin'] = titanic['Cabin'].notna().astype(int)

# 3. Ticket class indicators
titanic['IsFirstClass'] = (titanic['Pclass'] == 1).astype(int)
titanic['IsThirdClass'] = (titanic['Pclass'] == 3).astype(int)

# 4. Family survival strategy (medium families did better)
titanic['FamilySize_Category'] = pd.cut(titanic['FamilySize'], 
                                         bins=[0, 1, 4, 11], 
                                         labels=['Alone', 'Small', 'Large'])

# 5. Age * Class interaction (rich children vs poor children)
titanic['AgeClass'] = titanic['Age'] * titanic['Pclass']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Woman/Child policy
woc_survival = titanic.groupby('IsWomanOrChild')['Survived'].mean()
woc_survival.index = ['Adult Male', 'Woman or Child']
axes[0, 0].bar(woc_survival.index, woc_survival.values, color=['coral', 'lightblue'])
axes[0, 0].set_title('"Women and Children First" Policy', fontweight='bold')
axes[0, 0].set_ylim(0, 1)

# Has Cabin
cabin_survival = titanic.groupby('HasCabin')['Survived'].mean()
cabin_survival.index = ['No Cabin', 'Has Cabin']
axes[0, 1].bar(cabin_survival.index, cabin_survival.values, color=['salmon', 'gold'])
axes[0, 1].set_title('Having a Cabin Record', fontweight='bold')
axes[0, 1].set_ylim(0, 1)

# Family size category
fam_cat_survival = titanic.groupby('FamilySize_Category')['Survived'].mean()
axes[1, 0].bar(fam_cat_survival.index, fam_cat_survival.values, color='mediumorchid')
axes[1, 0].set_title('Family Size Category', fontweight='bold')
axes[1, 0].set_ylim(0, 1)

# Deck
deck_survival = titanic.groupby('Deck')['Survived'].mean().sort_values(ascending=False)
axes[1, 1].bar(deck_survival.index, deck_survival.values, color='teal')
axes[1, 1].set_title('Survival by Deck', fontweight='bold')
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("✅ Domain features leverage real-world knowledge!")
print(f"   Woman/Child survival rate: {titanic[titanic['IsWomanOrChild']==1]['Survived'].mean():.3f}")
print(f"   Adult male survival rate: {titanic[titanic['IsWomanOrChild']==0]['Survived'].mean():.3f}")


## 8. Feature Selection Techniques

💡 **Why This Feature Helps:** More features aren't always better. Irrelevant features add noise, increase training time, and cause overfitting. Feature selection keeps only the most predictive features.


In [ ]:
# Prepare engineered Titanic dataset for feature selection
titanic_engineered = titanic.copy()

# Encode categorical features
titanic_engineered['Sex'] = titanic_engineered['Sex'].map({'male': 0, 'female': 1})
titanic_engineered['Embarked'] = titanic_engineered['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
titanic_engineered['Title_Grouped'] = titanic_engineered['Title_Grouped'].map({
    'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4
})
titanic_engineered['FamilySize_Category'] = titanic_engineered['FamilySize_Category'].map({
    'Alone': 0, 'Small': 1, 'Large': 2
})
titanic_engineered['Deck'] = titanic_engineered['Deck'].astype('category').cat.codes

feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
                'FamilySize', 'IsAlone', 'FarePerPerson', 'Name_Length', 
                'Title_Grouped', 'IsWomanOrChild', 'HasCabin', 'IsFirstClass',
'IsThirdClass', 'AgeClass', 'FamilySize_Category', 'Deck']

X_eng = titanic_engineered[feature_cols].fillna(titanic_engineered[feature_cols].median())
y_eng = titanic_engineered['Survived']

# 1. Filter Method: Correlation with target
correlations = X_eng.corrwith(y_eng).abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Correlation plot
correlations.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Filter Method: Correlation with Target', fontweight='bold')
axes[0].set_xlabel('Absolute Correlation')

# 2. Wrapper Method: Recursive Feature Elimination
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=rf, n_features_to_select=10)
rfe.fit(X_eng, y_eng)
rfe_ranking = pd.Series(rfe.ranking_, index=feature_cols).sort_values()

rfe_ranking.plot(kind='barh', ax=axes[1], color='darkgreen')
axes[1].set_title('Wrapper Method: RFE Ranking (1=Selected)', fontweight='bold')
axes[1].set_xlabel('RFE Rank')

# 3. Embedded Method: Feature Importance from Random Forest
rf.fit(X_eng, y_eng)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

importances.plot(kind='barh', ax=axes[2], color='darkorange')
axes[2].set_title('Embedded Method: RF Feature Importance', fontweight='bold')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("✅ Top features by correlation:", correlations.head(5).index.tolist())
print("✅ Top features by RFE:", rfe_ranking[rfe_ranking == 1].index.tolist())
print("✅ Top features by importance:", importances.head(5).index.tolist())


## 9. End-to-End Feature Engineering Pipeline Example

Now let's put it all together and compare baseline vs. fully engineered features!


In [ ]:
# Complete Titanic Feature Engineering Pipeline

def engineer_titanic_features(df):
    """Complete feature engineering pipeline for Titanic dataset."""
    data = df.copy()
    
    # Fill missing values
    data['Age'] = data['Age'].fillna(data['Age'].median())
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
    data['Fare'] = data['Fare'].fillna(data['Fare'].median())
    
    # Numerical features
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    data['AgeBin'] = pd.cut(data['Age'], bins=[0, 12, 18, 35, 60, 100], labels=[0, 1, 2, 3, 4]).astype(int)
    
    # Categorical encoding
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    data['Embarked'] = data['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    
    # Text features
    data['Name_Length'] = data['Name'].str.len()
    data['Title'] = data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    title_map = {'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3}
    data['Title'] = data['Title'].map(title_map).fillna(4)  # 4 for rare titles
    
    # Domain features
    data['IsWomanOrChild'] = ((data['Sex'] == 1) | (data['Age'] < 16)).astype(int)
    data['HasCabin'] = data['Cabin'].notna().astype(int)
    data['IsFirstClass'] = (data['Pclass'] == 1).astype(int)
    data['AgeClass'] = data['Age'] * data['Pclass']
    
    # Select final features
    final_features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
                      'Embarked', 'FamilySize', 'IsAlone', 'FarePerPerson',
                      'AgeBin', 'Name_Length', 'Title', 'IsWomanOrChild',
                      'HasCabin', 'IsFirstClass', 'AgeClass']
    
    return data[final_features]

# Apply pipeline
X_full = engineer_titanic_features(titanic)
y_full = titanic['Survived']

print(f"📊 Engineered feature matrix shape: {X_full.shape}")
print(f"📋 Features: {X_full.columns.tolist()}")


In [ ]:
# Performance comparison: Baseline vs Engineered

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap before
corr_before = X_raw.corrwith(y).abs().sort_values(ascending=True)
axes[0].barh(corr_before.index, corr_before.values, color='lightcoral')
axes[0].set_title('Before: Feature Correlations with Target', fontweight='bold')
axes[0].set_xlim(0, 0.6)

# Correlation heatmap after
corr_after = X_full.corrwith(y_full).abs().sort_values(ascending=True)
axes[1].barh(corr_after.index, corr_after.values, color='mediumseagreen')
axes[1].set_title('After: Feature Correlations with Target', fontweight='bold')
axes[1].set_xlim(0, 0.6)

plt.tight_layout()
plt.show()

# Model comparison
print("\n" + "="*60)
print("📊 FINAL PERFORMANCE COMPARISON")
print("="*60)

results = []
for name, model in models.items():
    # Baseline
    base_score = cross_val_score(model, X_raw, y, cv=5, scoring='accuracy').mean()
    # Engineered
    eng_score = cross_val_score(model, X_full, y_full, cv=5, scoring='accuracy').mean()
    improvement = ((eng_score - base_score) / base_score) * 100
    
    results.append({
        'Model': name,
        'Baseline': f"{base_score:.4f}",
        'Engineered': f"{eng_score:.4f}",
'Improvement': f"{improvement:+.1f}%"
    })
    
    print(f"\n🤖 {name}:")
    print(f"   Baseline:    {base_score:.4f}")
    print(f"   Engineered:  {eng_score:.4f}")
    print(f"   Improvement: {improvement:+.1f}%")

print("\n" + "="*60)
print("✅ Feature engineering provides consistent, significant improvements!")


## 🛠️ Hands-On Exercises

Now it's your turn! Complete these exercises to solidify your feature engineering skills.

### Exercise 1: Create Interaction Features
Create at least 3 new interaction features for the House Prices dataset (e.g., `TotalSF` = `GrLivArea` + `TotalBsmtSF`, `HouseAge` = `YrSold` - `YearBuilt`). Visualize their relationship with `SalePrice`.


In [ ]:
n
# Exercise 1: Create interaction features for House Prices
# Hint: Combine existing features to create new meaningful ones

# Your code here:



### Exercise 2: Apply Different Encoding Strategies
Compare One-Hot Encoding, Target Encoding, and Frequency Encoding on the `Neighborhood` feature in House Prices. Train a simple model with each and compare RMSE scores.


In [ ]:
n
# Exercise 2: Compare encoding strategies
# Hint: Create three versions of the dataset with different encodings

# Your code here:



### Exercise 3: Engineer Domain-Specific Features for Titanic
Create the following domain-specific features and evaluate their impact:
- `TicketPrefix`: Extract prefix from Ticket (e.g., 'PC', 'STON')
- `FareCategory`: Bin Fare into economic classes (Low, Medium, High, Luxury)
- `Mother`: Identify mothers (female, has children, age > 18)


In [ ]:
n
# Exercise 3: Domain-specific Titanic features
# Hint: Use string operations and conditional logic

# Your code here:



### Exercise 4: Build a Complete Feature Engineering Pipeline
Create a function that takes the raw House Prices dataframe and returns a fully engineered feature matrix. Include at least 5 different types of feature engineering techniques we've covered.


In [ ]:
n
# Exercise 4: Complete House Prices pipeline
# Hint: Combine numerical, categorical, and domain features

# Your code here:



### Exercise 5: Measure Performance Improvement
Train a Random Forest model on raw House Prices features (minimal preprocessing) vs. your engineered features. Compare RMSE and R² scores using 5-fold cross-validation.


In [ ]:
n
# Exercise 5: Performance measurement
# Hint: Use cross_val_score with 'neg_root_mean_squared_error'

# Your code here:



### Exercise 6: Implement Feature Selection
Use at least two feature selection methods (Filter + Embedded) on your engineered House Prices dataset. Compare which features are selected by each method and train models with the selected features.


In [ ]:
n
# Exercise 6: Feature selection
# Hint: Use SelectKBest and SelectFromModel

# Your code here:



### Exercise 7: Polynomial Features Experiment
Create polynomial features (degree 2 and 3) for the top 3 numerical features in House Prices. Compare model performance and watch for overfitting.


In [ ]:
n
# Exercise 7: Polynomial features
# Hint: Use PolynomialFeatures and compare train vs test scores

# Your code here:



### Exercise 8: Date Feature Engineering
If you have a dataset with dates, create at least 5 date-derived features (year, month, day, quarter, is_weekend, days_since_start). Show how they improve a model's performance.


In [ ]:
n
# Exercise 8: Date features
# Hint: Use pandas datetime accessors

# Your code here:



### Exercise 9: Text Feature Extraction
Extract features from the Titanic `Cabin` column: deck letter, cabin number (if available), and whether the passenger had multiple cabins. Analyze their predictive power.


In [ ]:
n
# Exercise 9: Text features from Cabin
# Hint: Use string splitting and regex

# Your code here:



### Exercise 10: Full Kaggle-Style Pipeline
Combine everything! Create a production-ready feature engineering pipeline that:
1. Handles missing values intelligently
2. Creates 10+ new features
3. Encodes categoricals appropriately
4. Selects the best features
5. Outputs a submission-ready dataframe

Test it on both Titanic and House Prices datasets.


In [ ]:
n
# Exercise 10: Production pipeline
# Hint: Create a class or function with modular steps

# Your code here:



## Solutions & Best Practices (Review After Attempting)

Below are detailed solutions and explanations for each exercise. **Try the exercises first before peeking!**

---

### Solution 1: Interaction Features

```python
# Create meaningful interaction features
house['TotalSF'] = house['GrLivArea'] + house['TotalBsmtSF']
house['HouseAge'] = house['YrSold'] - house['YearBuilt']
house['RemodAge'] = house['YrSold'] - house['YearRemodAdd']
house['TotalBathrooms'] = (house['FullBath'] + 0.5 * house['HalfBath'] + 
                           house['BsmtFullBath'] + 0.5 * house['BsmtHalfBath'])
house['PorchSF'] = (house['OpenPorchSF'] + house['EnclosedPorch'] + 
                    house['3SsnPorch'] + house['ScreenPorch'])

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ['TotalSF', 'HouseAge', 'TotalBathrooms', 'PorchSF']
for idx, feat in enumerate(features):
    ax = axes[idx // 2, idx % 2]
    ax.scatter(house[feat], house['SalePrice'], alpha=0.5)
    ax.set_xlabel(feat)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'{feat} vs SalePrice')
plt.tight_layout()
plt.show()
```

**Key Insight:** `TotalSF` and `TotalBathrooms` typically show strong linear relationships with price. `HouseAge` often shows a negative correlation — older houses sell for less.

---

### Solution 2: Encoding Strategies Comparison

```python
from sklearn.ensemble import GradientBoostingRegressor

def compare_encodings(house_df):
    base_features = ['GrLivArea', 'TotalBsmtSF', 'YearBuilt']
    
    # One-Hot Encoding
    house_ohe = house_df.copy()
    ohe = pd.get_dummies(house_ohe['Neighborhood'], prefix='Neighborhood')
    house_ohe = pd.concat([house_ohe[base_features], ohe], axis=1)
    
    # Target Encoding (with smoothing to prevent overfitting)
    house_te = house_df.copy()
    global_mean = house_te['SalePrice'].mean()
    smoothing = 10
    agg = house_te.groupby('Neighborhood')['SalePrice'].agg(['mean', 'count'])
    agg['smooth'] = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    house_te['Neighborhood_TE'] = house_te['Neighborhood'].map(agg['smooth'])
    
    # Frequency Encoding
    house_fe = house_df.copy()
    freq_map = house_fe['Neighborhood'].value_counts().to_dict()
    house_fe['Neighborhood_FE'] = house_fe['Neighborhood'].map(freq_map)
    
    # Compare
    results = {}
    for name, df in [('One-Hot', house_ohe), ('Target Enc', house_te), ('Freq Enc', house_fe)]:
        X = df.drop(['SalePrice', 'Neighborhood'], axis=1, errors='ignore').select_dtypes(include=[np.number]).fillna(0)
        y = np.log1p(df['SalePrice'])
        model = GradientBoostingRegressor(random_state=42)
        score = -cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error').mean()
        results[name] = score
    
    return results

# print(compare_encodings(house))
```

**Key Insight:** One-Hot is safest but creates many columns. Target encoding is powerful but requires careful regularization (smoothing) to prevent data leakage. Frequency encoding is simple but often less predictive.

---

### Solution 3: Domain-Specific Titanic Features

```python
# Ticket Prefix extraction
titanic['TicketPrefix'] = titanic['Ticket'].str.extract(r'^([A-Za-z\.\/]+)', expand=False)
titanic['TicketPrefix'] = titanic['TicketPrefix'].fillna('None')
titanic['HasTicketPrefix'] = (titanic['TicketPrefix'] != 'None').astype(int)

# Fare categories
titanic['FareCategory'] = pd.qcut(titanic['Fare'], q=4, labels=['Low', 'Medium', 'High', 'Luxury'])

# Mother identification
titanic['IsMother'] = ((titanic['Sex'] == 'female') & 
                        (titanic['Parch'] > 0) & 
                        (titanic['Age'] > 18)).astype(int)

# Evaluate impact
print(titanic.groupby('FareCategory')['Survived'].mean())
print(f"Mother survival rate: {titanic[titanic['IsMother']==1]['Survived'].mean():.3f}")
```

**Key Insight:** `IsMother` is a powerful feature because mothers were prioritized for lifeboats. `FareCategory` captures economic status which correlated with deck location and survival access.

---

### Solution 4: Complete House Prices Pipeline

```python
def engineer_house_features(df):
    data = df.copy()
    
    # 1. Handle missing values
    data['LotFrontage'] = data.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median()))
    for col in ['GarageYrBlt', 'GarageArea', 'GarageCars']:
        data[col] = data[col].fillna(0)
    data['MasVnrArea'] = data['MasVnrArea'].fillna(0)
    
    # 2. Numerical interactions
    data['TotalSF'] = data['TotalBsmtSF'] + data['1stFlrSF'] + data['2ndFlrSF']
    data['TotalBathrooms'] = (data['FullBath'] + 0.5*data['HalfBath'] + 
                              data['BsmtFullBath'] + 0.5*data['BsmtHalfBath'])
    data['HouseAge'] = data['YrSold'] - data['YearBuilt']
    data['RemodAge'] = data['YrSold'] - data['YearRemodAdd']
    data['LotRatio'] = data['GrLivArea'] / data['LotArea']
    
    # 3. Binning
    data['HouseAgeBin'] = pd.cut(data['HouseAge'], bins=[-np.inf, 5, 20, 50, np.inf], 
                                  labels=['New', 'Recent', 'Old', 'VeryOld'])
    
    # 4. Categorical encoding
    # Quality as ordinal
    qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    for col in ['ExterQual', 'ExterCond', 'BsmtQual', 'KitchenQual']:
        data[col + '_Num'] = data[col].map(qual_map).fillna(0)
    
    # 5. Domain features
    data['HasPool'] = (data['PoolArea'] > 0).astype(int)
    data['Has2ndFloor'] = (data['2ndFlrSF'] > 0).astype(int)
    data['HasGarage'] = (data['GarageArea'] > 0).astype(int)
    data['HasFireplace'] = (data['Fireplaces'] > 0).astype(int)
    data['HasBasement'] = (data['TotalBsmtSF'] > 0).astype(int)
    
    # 6. Log transform skewed features
    skewed = ['GrLivArea', 'TotalSF', 'LotArea']
    for col in skewed:
        data[f'Log{col}'] = np.log1p(data[col])
    
    return data

# house_engineered = engineer_house_features(house)
```

**Key Insight:** A good pipeline is modular, handles missing values intelligently (groupby imputation), and creates features at multiple levels (interactions, binning, domain knowledge).

---

### Solution 5: Performance Measurement

```python
from sklearn.metrics import r2_score

# Raw features (minimal preprocessing)
raw_numeric = house.select_dtypes(include=[np.number]).fillna(0)
X_raw_house = raw_numeric.drop(['SalePrice', 'Id'], axis=1, errors='ignore')
y_house = np.log1p(house['SalePrice'])

# Engineered features
house_eng = engineer_house_features(house)
eng_numeric = house_eng.select_dtypes(include=[np.number]).fillna(0)
X_eng_house = eng_numeric.drop(['SalePrice', 'Id'], axis=1, errors='ignore')

# Compare
rf = RandomForestRegressor(n_estimators=100, random_state=42)

for name, X in [('Raw', X_raw_house), ('Engineered', X_eng_house)]:
    rmse = -cross_val_score(rf, X, y_house, cv=5, scoring='neg_root_mean_squared_error').mean()
    r2 = cross_val_score(rf, X, y_house, cv=5, scoring='r2').mean()
    print(f"{name}: RMSE={rmse:.4f}, R²={r2:.4f}")
```

**Key Insight:** You should see 10-20% improvement in RMSE with engineered features. The exact improvement depends on which features you created.

---

### Solution 6: Feature Selection

```python
# Filter method: Correlation
corr_threshold = 0.1
correlations = X_eng_house.corrwith(y_house).abs()
filter_features = correlations[correlations > corr_threshold].index.tolist()

# Embedded method: Random Forest importance
rf.fit(X_eng_house, y_house)
importances = pd.Series(rf.feature_importances_, index=X_eng_house.columns)
embedded_features = importances.nlargest(20).index.tolist()

# Compare selected features
print(f"Filter selected: {len(filter_features)} features")
print(f"Embedded selected: {len(embedded_features)} features")
print(f"Common features: {len(set(filter_features) & set(embedded_features))}")

# Train with selected features
for method, features in [('Filter', filter_features), ('Embedded', embedded_features)]:
    X_sel = X_eng_house[features]
    rmse = -cross_val_score(rf, X_sel, y_house, cv=5, scoring='neg_root_mean_squared_error').mean()
    print(f"{method}: RMSE={rmse:.4f} with {len(features)} features")
```

**Key Insight:** Filter methods are fast but may miss feature interactions. Embedded methods capture interactions but may overfit. Combining both approaches often works best.

---

### Solution 7: Polynomial Features

```python
from sklearn.preprocessing import PolynomialFeatures

top_features = ['GrLivArea', 'TotalBsmtSF', 'OverallQual']
X_poly_base = house[top_features].fillna(0)

for degree in [2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X_poly_base)
    
    # Split to check overfitting
    X_train, X_test, y_train, y_test = train_test_split(
        X_poly, y_house, test_size=0.2, random_state=42)
    
    model = Ridge(alpha=1.0)
    model.fit(X_train, y_train)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    print(f"Degree {degree}: Features={X_poly.shape[1]}, "
          f"Train RMSE={train_rmse:.4f}, Test RMSE={test_rmse:.4f}")
```

**Key Insight:** Degree 2 often improves performance. Degree 3 may cause overfitting (train RMSE much lower than test RMSE). Always monitor the gap between train and test performance!

---

### Solution 8: Date Features

```python
# Assuming df has a 'Date' column
def extract_date_features(df, date_col):
    data = df.copy()
    data['Year'] = data[date_col].dt.year
    data['Month'] = data[date_col].dt.month
    data['Day'] = data[date_col].dt.day
    data['Quarter'] = data[date_col].dt.quarter
    data['DayOfWeek'] = data[date_col].dt.dayofweek
    data['IsWeekend'] = (data['DayOfWeek'] >= 5).astype(int)
    data['IsMonthStart'] = data[date_col].dt.is_month_start.astype(int)
    data['IsMonthEnd'] = data[date_col].dt.is_month_end.astype(int)
    data['DaysSinceStart'] = (data[date_col] - data[date_col].min()).dt.days
    
    # Cyclical encoding
    data['Month_Sin'] = np.sin(2 * np.pi * data['Month'] / 12)
    data['Month_Cos'] = np.cos(2 * np.pi * data['Month'] / 12)
    data['DayOfWeek_Sin'] = np.sin(2 * np.pi * data['DayOfWeek'] / 7)
    data['DayOfWeek_Cos'] = np.cos(2 * np.pi * data['DayOfWeek'] / 7)
    
    return data

# Usage: df = extract_date_features(df, 'SaleDate')
```

**Key Insight:** Cyclical encoding is crucial for month/day features. Without it, January (1) and December (12) would appear far apart in numeric space, which is incorrect.

---

### Solution 9: Text Features from Cabin

```python
# Extract Cabin features
titanic['CabinDeck'] = titanic['Cabin'].str[0]
titanic['CabinDeck'] = titanic['CabinDeck'].fillna('U')  # Unknown

# Extract cabin number (if present)
titanic['CabinNum'] = titanic['Cabin'].str.extract(r'(\d+)', expand=False)
titanic['CabinNum'] = pd.to_numeric(titanic['CabinNum'], errors='coerce')
titanic['HasCabinNum'] = titanic['CabinNum'].notna().astype(int)

# Multiple cabins
titanic['NumCabins'] = titanic['Cabin'].str.count(' ') + 1
titanic['NumCabins'] = titanic['NumCabins'].fillna(0)
titanic['HasMultipleCabins'] = (titanic['NumCabins'] > 1).astype(int)

# Analyze
print(titanic.groupby('CabinDeck')['Survived'].mean().sort_values(ascending=False))
print(f"Multiple cabins survival: {titanic[titanic['HasMultipleCabins']==1]['Survived'].mean():.3f}")
```

**Key Insight:** Cabin deck B and C had higher survival rates (higher decks, closer to lifeboats). Multiple cabins indicate wealth and potentially better access to survival resources.

---

### Solution 10: Production Pipeline

```python
class FeatureEngineeringPipeline:
    def __init__(self):
        self.scaler = StandardScaler()
        self.feature_names = None
    
    def fit_transform(self, df, target_col=None):
        data = df.copy()
        
        # Step 1: Missing values
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        data[numeric_cols] = data[numeric_cols].fillna(data[numeric_cols].median())
        
        # Step 2: Feature creation
        if 'GrLivArea' in data.columns and 'TotalBsmtSF' in data.columns:
            data['TotalSF'] = data['GrLivArea'] + data['TotalBsmtSF']
        if 'YearBuilt' in data.columns and 'YrSold' in data.columns:
            data['HouseAge'] = data['YrSold'] - data['YearBuilt']
        
        # Step 3: Encoding
        categorical = data.select_dtypes(include=['object']).columns
        data = pd.get_dummies(data, columns=categorical, drop_first=True)
        
        # Step 4: Scaling
        numeric_features = data.select_dtypes(include=[np.number]).columns
        if target_col and target_col in numeric_features:
            numeric_features = numeric_features.drop(target_col)
        data[numeric_features] = self.scaler.fit_transform(data[numeric_features])
        
        self.feature_names = data.columns.tolist()
        return data
    
    def transform(self, df):
        # Same steps but using fitted scaler
        pass

# pipeline = FeatureEngineeringPipeline()
# processed = pipeline.fit_transform(house, target_col='SalePrice')
```

**Key Insight:** Production pipelines should be class-based with separate `fit_transform` and `transform` methods to prevent data leakage between train and test sets.

---

## 🎯 Best Practices Summary

1. **Always validate with cross-validation** — Don't trust a single train/test split
2. **Monitor for data leakage** — Never use target variable information inappropriately
3. **Start with EDA** — Understand your data before engineering features
4. **Document everything** — Every feature should have a clear business justification
5. **Test incrementally** — Add features one at a time and measure impact
6. **Beware of the curse of dimensionality** — More features ≠ better performance
7. **Use domain knowledge** — The best features come from understanding the problem
8. **Handle missing values carefully** — Different strategies for different feature types
9. **Normalize/Scale appropriately** — Required for many algorithms, optional for tree-based models
10. **Keep it reproducible** — Your pipeline should work identically on new data


## 🎓 Conclusion

You now have strong feature engineering skills — often the difference between mediocre and winning models. 

Today you learned:
- ✅ Why feature engineering matters more than algorithm selection
- ✅ How to create powerful numerical features (binning, scaling, transforms, interactions, polynomials)
- ✅ How to encode categorical variables effectively
- ✅ How to extract value from dates, text, and domain knowledge
- ✅ How to select the best features and avoid overfitting
- ✅ How to build production-ready feature engineering pipelines

**Tomorrow we explore Advanced Model Evaluation & Ensembling Techniques** — where we'll learn how to combine multiple models and evaluate them like a Kaggle master! 🚀

---

*Python & AI Learning Path | Day 62 / 369* ⭐
